# 02 — Image Compression

## Overview

This notebook implements the second stage of **Image Compression Effects on Face Recognition Fairness**.

It takes the standardized **L0 baseline facial images** produced by `01_image_preprocessing.ipynb` and generates compressed probe images using three codecs:

- **JPEG**
- **JPEG XL**
- **HEIC**

Each codec is evaluated at five compression levels:

- **L1** — lowest compression strength / highest retained quality in this experiment
- **L2**
- **L3**
- **L4**
- **L5** — strongest compression / lowest retained quality in this experiment

### Pipeline

L0 preprocessed PNG images  
→ JPEG / JPEG XL / HEIC compression  
→ L1–L5 compressed probe images  
→ embedding extraction in `03_embedding_extraction.ipynb`

> **Research preservation:** The codec parameters below are the values used in the original experiment. They are preserved exactly and should not be changed when reproducing the reported results.

> **Privacy:** These images contain participant faces. Baseline and compressed images must remain outside the public GitHub repository.

## Environment Setup

This notebook uses external command-line tools for JPEG XL and HEIC compression.

The original experiment was executed in Google Colab and required:

- **FFmpeg** for multimedia codec support
- **libheif-examples** for `heif-enc` and `heif-convert`
- **libx265-dev** for HEVC-related support
- **JPEG XL v0.11.1** tools (`cjxl` and `djxl`)

The installation steps below are retained from the original experimental environment to support reproducibility.

In [ ]:
# HEIC / FFmpeg dependencies
print("--- Installing FFmpeg and libheif-examples for HEIC support ---")

!apt-get update
!apt-get install -y ffmpeg libheif-examples

print("✅ FFmpeg and libheif-examples installed.")

In [ ]:
# HEVC support
print("--- Installing libx265-dev for HEVC support ---")

!apt-get update
!apt-get install -y libx265-dev

print("✅ libx265-dev installed.")

## Library Imports

The compression workflow uses:

- `os` and `shutil` for file and directory handling
- `subprocess` for running external codec tools
- `tarfile` and `requests` for installing the JPEG XL binaries
- OpenCV for reading and writing PNG/JPEG images

In [ ]:
import os
import shutil
import subprocess
import tarfile

import cv2
import requests

## JPEG XL Tool Setup

The original notebook used **JPEG XL v0.11.1**.

The static release archive is downloaded from the official `libjxl` GitHub release and the `cjxl` and `djxl` binaries are copied to `/usr/local/bin/`.

- `cjxl` is used to create `.jxl` files.
- `djxl` is only needed when visually decoding JPEG XL files for inspection.

This setup does not alter the JPEG XL distance values used later in the experiment.

In [ ]:
# JPEG XL version used in the original experiment
JXL_VERSION = "v0.11.1"
JXL_FILENAME = f"jxl-linux-x86_64-static-{JXL_VERSION}.tar.gz"
JXL_DOWNLOAD_URL = (
    f"https://github.com/libjxl/libjxl/releases/download/"
    f"{JXL_VERSION}/{JXL_FILENAME}"
)

JXL_INSTALL_DIR = "/content/jxl_install"
CJXL_BINARY = "/usr/local/bin/cjxl"
DJXL_BINARY = "/usr/local/bin/djxl"


def install_jxl_tools():
    """Install the JPEG XL command-line tools used by this notebook."""

    if not os.path.exists(JXL_INSTALL_DIR):
        print(f"Downloading JPEG XL {JXL_VERSION}...")

        response = requests.get(
            JXL_DOWNLOAD_URL,
            stream=True,
            timeout=120,
        )
        response.raise_for_status()

        with open(JXL_FILENAME, "wb") as file:
            file.write(response.content)

        os.makedirs(
            JXL_INSTALL_DIR,
            exist_ok=True,
        )

        with tarfile.open(
            JXL_FILENAME,
            "r:gz",
        ) as archive:
            archive.extractall(
                path=JXL_INSTALL_DIR,
            )

    tools_dir = os.path.join(
        JXL_INSTALL_DIR,
        "tools",
    )

    required_tools = {
        "cjxl": CJXL_BINARY,
        "djxl": DJXL_BINARY,
    }

    for tool_name, target_path in required_tools.items():
        source_path = os.path.join(
            tools_dir,
            tool_name,
        )

        # Fallback search in case the release archive layout differs.
        if not os.path.exists(source_path):
            source_path = None

            for root, _, files in os.walk(JXL_INSTALL_DIR):
                if tool_name in files:
                    source_path = os.path.join(
                        root,
                        tool_name,
                    )
                    break

        if source_path is None:
            raise FileNotFoundError(
                f"Could not locate {tool_name} "
                f"inside {JXL_INSTALL_DIR}."
            )

        shutil.copy2(
            source_path,
            target_path,
        )
        os.chmod(
            target_path,
            0o755,
        )

    print("JPEG XL tools are ready.")


install_jxl_tools()

## Dataset Paths and Compression Configuration

### Input

The input directory contains the **112 × 112 PNG L0 baseline images** generated by Notebook 01.

### Output

Compressed probe images are organized by:

```text
compressed_images/
├── jpeg/
│   ├── L1/
│   ├── ...
│   └── L5/
├── jxl/
│   ├── L1/
│   ├── ...
│   └── L5/
└── heic/
    ├── L1/
    ├── ...
    └── L5/
```

Each level is further divided into the Malay, Chinese, and Indian demographic folders.

### Compression parameters

The following parameter values are copied directly from the original experimental notebook and are **not recalibrated or replaced** here.

| Level | JPEG quality | JPEG XL distance | HEIC control value |
|---|---:|---:|---:|
| L1 | 98 | 0.23 | 0 |
| L2 | 96 | 0.40 | 8 |
| L3 | 80 | 1.40 | 18 |
| L4 | 20 | 6.62 | 28 |
| L5 | 5 | 9.95 | 34 |

For HEIC, the control value is passed through the same inverse mapping to `heif-enc` quality that was used in the original implementation.

In [ ]:
# Repository-relative paths.
# These private image directories should remain excluded by .gitignore.
INPUT_DIR = "../data/preprocessed_images"
PROBE_DIR = "../data/compressed_images"

# Demographic group IDs used in output filenames.
ETHNIC_MAP = {
    "malay": "01",
    "chinese": "02",
    "indian": "03",
}

# Compression settings used in the original experiment.
SETTINGS = {
    "L1": {
        "jpeg": 98,
        "jxl": 0.23,
        "heic": 0,
    },
    "L2": {
        "jpeg": 96,
        "jxl": 0.40,
        "heic": 8,
    },
    "L3": {
        "jpeg": 80,
        "jxl": 1.40,
        "heic": 18,
    },
    "L4": {
        "jpeg": 20,
        "jxl": 6.62,
        "heic": 28,
    },
    "L5": {
        "jpeg": 5,
        "jxl": 9.95,
        "heic": 34,
    },
}

## Initialize Compression Output Directories

A full experimental run starts from a clean output directory so that files from previous runs cannot be mixed with newly generated probe images.

This preserves the behavior of the original notebook, which deleted the previous probe directory before creating the complete codec/level/demographic folder structure.

> **Caution:** Running this cell deletes the existing `compressed_images/` directory.

In [ ]:
print(f"Resetting compression output directory: {PROBE_DIR}")

if os.path.exists(PROBE_DIR):
    shutil.rmtree(PROBE_DIR)

for codec in ["jpeg", "jxl", "heic"]:
    for level in SETTINGS:
        for ethnicity in ETHNIC_MAP:
            os.makedirs(
                os.path.join(
                    PROBE_DIR,
                    codec,
                    level,
                    ethnicity,
                ),
                exist_ok=True,
            )

print("Compression output directories initialized.")

## JPEG Compression

JPEG compression is performed with OpenCV using `cv2.IMWRITE_JPEG_QUALITY`.

For every demographic group and every L0 PNG image:

1. Load the baseline image.
2. Apply each L1–L5 JPEG quality value.
3. Save the compressed file as `.jpg`.

### Output naming

```text
L<level>_<demographic_id>_<image_id>.jpg
```

Example:

```text
L3_01_0001.jpg
```

In [ ]:
print("--- Starting JPEG Compression ---")

for ethnicity, ethnicity_id in ETHNIC_MAP.items():
    input_path = os.path.join(
        INPUT_DIR,
        ethnicity,
    )

    for filename in os.listdir(input_path):
        if not filename.lower().endswith(".png"):
            continue

        image_path = os.path.join(
            input_path,
            filename,
        )

        image = cv2.imread(image_path)

        if image is None:
            raise ValueError(
                f"Could not read baseline image: {image_path}"
            )

        file_idx = (
            filename
            .split("_")[-1]
            .replace(".png", "")
        )

        for level, parameters in SETTINGS.items():
            output_name = (
                f"{level}_{ethnicity_id}_{file_idx}.jpg"
            )

            output_path = os.path.join(
                PROBE_DIR,
                "jpeg",
                level,
                ethnicity,
                output_name,
            )

            cv2.imwrite(
                output_path,
                image,
                [
                    cv2.IMWRITE_JPEG_QUALITY,
                    parameters["jpeg"],
                ],
            )

print("JPEG compression complete.")

## JPEG XL Compression

JPEG XL compression uses the `cjxl` command-line encoder.

The original experiment controls JPEG XL degradation using the `-d` distance parameter. The exact L1–L5 distance values are preserved.

For each L0 image:

```text
cjxl input.png output.jxl -d <distance> --quiet
```

The resulting `.jxl` files become JPEG XL probe images for the embedding-extraction stage.

In [ ]:
print("--- Starting JPEG XL Compression ---")

for ethnicity, ethnicity_id in ETHNIC_MAP.items():
    input_path = os.path.join(
        INPUT_DIR,
        ethnicity,
    )

    for filename in os.listdir(input_path):
        if not filename.lower().endswith(".png"):
            continue

        image_path = os.path.join(
            input_path,
            filename,
        )

        file_idx = (
            filename
            .split("_")[-1]
            .replace(".png", "")
        )

        for level, parameters in SETTINGS.items():
            output_name = (
                f"{level}_{ethnicity_id}_{file_idx}.jxl"
            )

            output_path = os.path.join(
                PROBE_DIR,
                "jxl",
                level,
                ethnicity,
                output_name,
            )

            command = [
                CJXL_BINARY,
                image_path,
                output_path,
                "-d",
                str(parameters["jxl"]),
                "--quiet",
            ]

            subprocess.run(
                command,
                check=True,
            )

print("JPEG XL compression complete.")

## HEIC Compression

HEIC compression is performed with `heif-enc`.

The original notebook stored HEIC level values on a CRF-like 0–51 scale and converted them to the 0–100 quality scale expected by `heif-enc`.

The original mapping is preserved:

```text
heif_quality = (51 - control_value) × (100 / 51)
```

The result is clamped to the valid 0–100 range and converted to an integer.

No attempt is made here to redefine or scientifically recalibrate these levels.

In [ ]:
def compress_png_to_heic(
    input_png_path,
    output_heic_path,
    quality,
):
    """
    Compress a PNG image to HEIC using the mapping from the
    original experiment.

    Parameters
    ----------
    input_png_path : str
        Path to the source L0 PNG image.
    output_heic_path : str
        Destination path for the HEIC probe image.
    quality : int
        Original HEIC control value stored in SETTINGS.
    """

    # Preserve the original inverse mapping to heif-enc quality.
    heif_quality = max(
        0,
        min(
            100,
            int(
                (51 - quality)
                * (100 / 51)
            ),
        ),
    )

    command = [
        "heif-enc",
        "-q",
        str(heif_quality),
        "-o",
        output_heic_path,
        input_png_path,
    ]

    try:
        subprocess.run(
            command,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            check=True,
        )

    except subprocess.CalledProcessError as error:
        raise RuntimeError(
            "HEIC compression failed for "
            f"{input_png_path}"
        ) from error

    except FileNotFoundError as error:
        raise FileNotFoundError(
            "heif-enc was not found. "
            "Install libheif-examples before running this stage."
        ) from error

In [ ]:
print("--- Starting HEIC Compression ---")

for ethnicity, ethnicity_id in ETHNIC_MAP.items():
    input_path = os.path.join(
        INPUT_DIR,
        ethnicity,
    )

    for filename in os.listdir(input_path):
        if not filename.lower().endswith(".png"):
            continue

        image_path = os.path.join(
            input_path,
            filename,
        )

        file_idx = (
            filename
            .split("_")[-1]
            .replace(".png", "")
        )

        for level, parameters in SETTINGS.items():
            output_name = (
                f"{level}_{ethnicity_id}_{file_idx}.heic"
            )

            output_path = os.path.join(
                PROBE_DIR,
                "heic",
                level,
                ethnicity,
                output_name,
            )

            compress_png_to_heic(
                image_path,
                output_path,
                quality=parameters["heic"],
            )

print("HEIC compression complete.")

## Output of This Stage

After a successful run, this notebook produces five compressed versions of every L0 baseline image for each codec.

### Experimental conditions

```text
L0 = original preprocessed PNG baseline

JPEG:
L1, L2, L3, L4, L5

JPEG XL:
L1, L2, L3, L4, L5

HEIC:
L1, L2, L3, L4, L5
```

These compressed probe images are consumed by **`03_embedding_extraction.ipynb`**, where pretrained face-recognition models extract embeddings from the baseline and compressed conditions.